# 05 架构 + 中间态输出

目标：观察模型结构、hidden states、attention 输出和 KV cache。注意 attention map 是 seq_len 平方级显存开销，默认关闭。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


## 1. 工具函数


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

OUTPUT_ATTENTIONS = False


def tensor_summary(value):
    if value is None:
        return "None"
    if hasattr(value, "shape"):
        return f"shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}"
    return type(value).__name__


def print_tensor_sequence(name, values):
    if values is None:
        print(f"{name}: None")
        return

    print(f"{name} count:", len(values))
    for index, value in enumerate(values):
        print(f"{name}[{index:02d}]:", tensor_summary(value))


def print_cache_summary(cache):
    if cache is None:
        print("KV cache: None")
        return

    print("KV cache type:", type(cache).__name__)

    try:
        print("KV cache layers:", len(cache))
    except TypeError:
        print("KV cache layers: unknown")

    if hasattr(cache, "get_seq_length"):
        print("KV cache sequence length:", cache.get_seq_length())

    if hasattr(cache, "key_cache") and getattr(cache, "key_cache"):
        print("layer0 key:", tensor_summary(cache.key_cache[0]))
        print("layer0 value:", tensor_summary(cache.value_cache[0]))
    elif isinstance(cache, (tuple, list)) and cache:
        first_layer = cache[0]
        if isinstance(first_layer, (tuple, list)) and len(first_layer) >= 2:
            print("layer0 key:", tensor_summary(first_layer[0]))
            print("layer0 value:", tensor_summary(first_layer[1]))


## 2. 加载模型并查看结构


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

print("=== Model Config ===")
print(model.config)

print("\n=== Model Architecture ===")
print(model)


## 3. forward 并输出中间态


In [ ]:
messages = [
    {"role": "user", "content": "用一句话解释 hidden state 是什么？"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=True,
        output_hidden_states=True,
        output_attentions=OUTPUT_ATTENTIONS,
        return_dict=True,
    )

last_token_logits = outputs.logits[:, -1, :]
next_token_id = last_token_logits.argmax(dim=-1)
next_token = tokenizer.decode(next_token_id)

print("\n=== Forward Outputs ===")
print("outputs type:", type(outputs).__name__)
print("outputs keys:", list(outputs.keys()))
print("input_ids:", tensor_summary(inputs["input_ids"]))
print("logits:", tensor_summary(outputs.logits))
print("last token logits:", tensor_summary(last_token_logits))
print("greedy next token id:", next_token_id.item())
print("greedy next token:", repr(next_token))


## 4. hidden states、attentions 和 KV cache


In [ ]:
print("\n=== Hidden States ===")
print_tensor_sequence("hidden_states", outputs.hidden_states)

print("\n=== Attentions ===")
print_tensor_sequence("attentions", outputs.attentions)

print("\n=== KV Cache ===")
print_cache_summary(outputs.past_key_values)
